# Swarm graph starter

Colab-ready smoke demo for **Graph Problem Swarm Intelligence**.

1. Install deps from `requirements.txt` (or the pin list below).
2. Build a small NetworkX graph.
3. Run a trivial swarm-style placeholder loop over nodes.

Open from GitHub:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gopalenparasuraman/graph-problem-swarm-intelligence/blob/main/notebooks/01_swarm_graph_starter.ipynb)


## Install (Colab)

If you opened this notebook from the GitHub badge, run the clone+install cell.  
If the notebook already lives next to `requirements.txt`, the local pip cell is enough.


In [ ]:
# Colab: clone repo and install requirements (skip if already local)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/gopalenparasuraman/graph-problem-swarm-intelligence.git"
ROOT = Path("/content/graph-problem-swarm-intelligence")

if IN_COLAB:
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
    os.chdir(ROOT)
    print("cwd:", Path.cwd())

req = Path("requirements.txt")
if req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "networkx", "matplotlib", "numpy"])
print("deps ready")


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

print("networkx", nx.__version__)
print("numpy", np.__version__)


## Graph + trivial swarm placeholder

Agents walk a random graph (Karate Club) and deposit a simple pheromone-like score on visited nodes. This is only a smoke test — replace with a real swarm heuristic later.


In [ ]:
G = nx.karate_club_graph()
rng = np.random.default_rng(42)

n_agents = 8
n_steps = 40
pheromone = {n: 0.0 for n in G.nodes}

positions = {i: int(rng.choice(list(G.nodes))) for i in range(n_agents)}

for _ in range(n_steps):
    for a, node in list(positions.items()):
        pheromone[node] += 1.0
        nbrs = list(G.neighbors(node))
        if not nbrs:
            continue
        # Prefer higher-pheromone neighbors with a little noise
        scores = np.array([pheromone[n] + 0.1 for n in nbrs], dtype=float)
        probs = scores / scores.sum()
        positions[a] = int(rng.choice(nbrs, p=probs))

top = sorted(pheromone.items(), key=lambda kv: kv[1], reverse=True)[:5]
print(f"graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("top pheromone nodes:", top)
assert G.number_of_nodes() == 34
assert sum(pheromone.values()) > 0
print("smoke OK")


In [ ]:
pos = nx.spring_layout(G, seed=42)
vals = np.array([pheromone[n] for n in G.nodes()])
fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx(
    G,
    pos=pos,
    ax=ax,
    node_color=vals,
    cmap=plt.cm.viridis,
    node_size=280,
    with_labels=True,
    font_size=8,
    edge_color="#cccccc",
)
ax.set_title("Karate Club — trivial swarm pheromone")
ax.axis("off")
plt.tight_layout()
plt.show()
